# System Stability

A linear time-invariant (LTI) system's natural behavior is governed entirely by its **poles** — roots of the characteristic polynomial. Their location in the complex plane determines whether the response decays, persists, or diverges.

## Continuous Time

Poles $p_i = \sigma_i + j\omega_i$ of the transfer function denominator $D(s) = \prod_i (s - p_i)$:

| Condition | Stability |
|-----------|-----------|
| All $\sigma_i < 0$ | **Asymptotically stable** — response decays to zero |
| No $\sigma_i > 0$, non-repeated $\sigma_i = 0$ | **Marginally stable** — bounded, non-decaying |
| Any $\sigma_i > 0$, or repeated $j\omega$-axis pole | **Unstable** — unbounded growth |

Stable region: the **open left half-plane** ($\text{Re}(s) < 0$, shaded green in the plot).

## Discrete Time

The mapping $z = e^{sT}$ transforms the left half-plane into the open unit disk. Poles $z_i$ of $D(z) = \prod_i (z - z_i)$:

| Condition | Stability |
|-----------|-----------|
| All $\|z_i\| < 1$ | **Asymptotically stable** |
| All $\|z_i\| \le 1$ and poles on $|z|=1$ are **simple** | **Marginally stable** |
| Any $\|z_i\| > 1$, or repeated pole on $|z|=1$ | **Unstable** |

Stable region: the **open unit disk** ($|z| < 1$).

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

def _verdict_ct(poles):
    tol = 1e-9
    reals = [p.real for p in poles]
    if any(r > tol for r in reals):
        return 'UNSTABLE', '#c0392b'
    if all(r < -tol for r in reals):
        return 'ASYMPTOTICALLY STABLE', '#27ae60'
    imag_only = [p for p in poles if abs(p.real) <= tol]
    imgs = [round(p.imag, 6) for p in imag_only]
    if len(imgs) != len(set(imgs)):
        return 'UNSTABLE (repeated jw pole)', '#c0392b'
    return 'MARGINALLY STABLE', '#e67e22'

def update_ct(r1, i1, r2, i2, r3, i3):
    poles = [complex(r1, i1), complex(r2, i2), complex(r3, i3)]
    verdict, vcol = _verdict_ct(poles)
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    ax.fill_betweenx([-2.5, 2.5], -2.5, 0, alpha=0.08, color='#2ecc71')
    ax.fill_betweenx([-2.5, 2.5],  0, 2.5, alpha=0.08, color='#e74c3c')
    ax.axhline(0, color='k', lw=0.8)
    ax.axvline(0, color='k', lw=0.8)
    for k, (p, col) in enumerate(zip(poles, ['#2c3e50', '#7f8c8d', '#aab7b8'])):
        ax.plot(p.real, p.imag, 'x', color=col, ms=14, mew=2.5,
                label=f'p{k+1} = {p.real:+.1f}{p.imag:+.1f}j')
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.set_xlabel('Real  \u03c3', fontsize=11)
    ax.set_ylabel('Imaginary  j\u03c9', fontsize=11)
    ax.set_title(f'CT Pole Map  \u00b7  {verdict}', color=vcol, fontweight='bold')
    ax.legend(fontsize=9, loc='lower right')
    ax.text(-2.4, 2.3, '\u2190 stable', fontsize=8, color='#27ae60')
    ax.text( 0.2, 2.3, 'unstable \u2192', fontsize=8, color='#c0392b')

    t = np.linspace(0, 6, 600)
    y = sum(np.exp(p.real * t) * np.cos(p.imag * t) for p in poles) / 3
    ax2.plot(t, np.clip(y, -20, 20), color='steelblue', lw=2)
    ax2.axhline(0, color='k', lw=0.5, ls='--')
    ax2.set_xlabel('Time  t', fontsize=11)
    ax2.set_ylabel('y(t)', fontsize=11)
    ax2.set_title(r'Natural Response  $y(t)=\sum_i e^{\sigma_i t}\cos(\omega_i t)$', fontsize=11)
    ax2.grid(True, alpha=0.3)
    if np.any(np.abs(y) > 20):
        ax2.text(0.5, 0.95, '(clipped for display)', transform=ax2.transAxes,
                 ha='center', fontsize=8, color='gray')
    plt.tight_layout()
    plt.show()

_sl = lambda d, v: widgets.FloatSlider(min=-2, max=2, step=0.1, value=v, description=d,
    continuous_update=False, style={'description_width': '65px'},
    layout=widgets.Layout(width='260px'))

interact(update_ct,
    r1=_sl('Re(p1)', -0.5), i1=_sl('Im(p1)',  1.0),
    r2=_sl('Re(p2)', -0.5), i2=_sl('Im(p2)', -1.0),
    r3=_sl('Re(p3)', -1.0), i3=_sl('Im(p3)',  0.0))

interactive(children=(FloatSlider(value=-0.5, continuous_update=False, description='Re(p1)', layout=Layout(wid…

<function __main__.update_ct(r1, i1, r2, i2, r3, i3)>

## Discrete-Time Pole Map

The stability boundary is the **unit circle**: poles inside ($|z|<1$) give decay, poles outside ($|z|>1$) give growth. A complex pole $z = r e^{j\theta}$ oscillates at normalized frequency $\theta$ rad/sample with amplitude envelope $r^k$.

Drag poles across the unit circle boundary and watch the impulse sequence $y[k]$ switch from convergent to divergent.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

def _verdict_dt(poles):
    tol = 1e-9
    mags = [abs(z) for z in poles]
    if any(m > 1 + tol for m in mags):
        return 'UNSTABLE', '#c0392b'
    if all(m < 1 - tol for m in mags):
        return 'ASYMPTOTICALLY STABLE', '#27ae60'
    unit = [z for z, m in zip(poles, mags) if abs(m - 1) <= tol]
    angles = [round(np.angle(z), 6) for z in unit]
    if len(angles) != len(set(angles)):
        return 'UNSTABLE (repeated unit-circle pole)', '#c0392b'
    return 'MARGINALLY STABLE', '#e67e22'

def update_dt(r1, i1, r2, i2, r3, i3):
    poles = [complex(r1, i1), complex(r2, i2), complex(r3, i3)]
    verdict, vcol = _verdict_dt(poles)
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    theta_c = np.linspace(0, 2 * np.pi, 300)
    ax.fill(np.cos(theta_c), np.sin(theta_c), alpha=0.08, color='#2ecc71')
    ax.plot(np.cos(theta_c), np.sin(theta_c), color='gray', lw=1, ls='--', label='unit circle')
    ax.axhline(0, color='k', lw=0.8)
    ax.axvline(0, color='k', lw=0.8)
    for k, (p, col) in enumerate(zip(poles, ['#2c3e50', '#7f8c8d', '#aab7b8'])):
        ax.plot(p.real, p.imag, 'x', color=col, ms=14, mew=2.5,
                label=f'z{k+1} = {p.real:+.2f}{p.imag:+.2f}j  |z|={abs(p):.2f}')
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
    ax.set_xlabel('Real', fontsize=11)
    ax.set_ylabel('Imaginary', fontsize=11)
    ax.set_title(f'DT Pole Map  \u00b7  {verdict}', color=vcol, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')

    k_arr = np.arange(0, 20)
    y = np.real(sum(p**k_arr for p in poles)) / 3
    yc = np.clip(y, -15, 15)
    ax2.vlines(k_arr, 0, yc, color='steelblue', alpha=0.7, lw=1.5)
    ax2.scatter(k_arr, yc, color='steelblue', zorder=3, s=30)
    ax2.axhline(0, color='k', lw=0.5, ls='--')
    ax2.set_xlabel('Sample  k', fontsize=11)
    ax2.set_ylabel('y[k]', fontsize=11)
    ax2.set_title(r'Natural Response  $y[k]=\mathrm{Re}\!\left(\sum_i z_i^k\right)$', fontsize=11)
    ax2.grid(True, alpha=0.3)
    if np.any(np.abs(y) > 15):
        ax2.text(0.5, 0.95, '(clipped)', transform=ax2.transAxes,
                 ha='center', fontsize=8, color='gray')
    plt.tight_layout()
    plt.show()

_sl2 = lambda d, v: widgets.FloatSlider(min=-2, max=2, step=0.05, value=v, description=d,
    continuous_update=False, style={'description_width': '65px'},
    layout=widgets.Layout(width='260px'))

interact(update_dt,
    r1=_sl2('Re(z1)',  0.6), i1=_sl2('Im(z1)',  0.6),
    r2=_sl2('Re(z2)',  0.6), i2=_sl2('Im(z2)', -0.6),
    r3=_sl2('Re(z3)', -0.7), i3=_sl2('Im(z3)',  0.0))

interactive(children=(FloatSlider(value=0.6, continuous_update=False, description='Re(z1)', layout=Layout(widt…

<function __main__.update_dt(r1, i1, r2, i2, r3, i3)>

## Natural Modes of Evolution

Each pole generates one **natural mode** — the elementary waveform that pole contributes to the total response $y = \sum_i A_i\,m_i$, where $A_i$ are set by initial conditions.

| Domain | Pole type | Mode $m_i$ | Decays when |
|--------|-----------|------------|-------------|
| **CT** | Simple real $p = \sigma$ | $e^{\sigma t}$ | $\sigma < 0$ |
| **CT** | Repeated real $p = \sigma$ | $t\,e^{\sigma t}$ | $\sigma < 0$ |
| **CT** | Complex pair $\sigma \pm j\omega$ | $e^{\sigma t}\cos(\omega t)$ | $\sigma < 0$ |
| **DT** | Simple real $z = \lambda$ | $\lambda^k$ | $|\lambda| < 1$ |
| **DT** | Repeated real $z = \lambda$ | $k\,\lambda^{k-1}$ | $|\lambda| < 1$ |
| **DT** | Complex pair $r e^{\pm j\theta}$ | $r^k \cos(\theta k)$ | $r < 1$ |

The sliders below drive **all three CT modes simultaneously** with the same $\sigma$ and $\omega$, making it clear that every term of the response shares the same stability character.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

def update_ct_modes(sigma, omega):
    t = np.linspace(0, 6, 600)
    m1 = np.exp(sigma * t)
    m2 = t * np.exp(sigma * t)
    m3 = np.exp(sigma * t) * np.cos(omega * t)
    env = np.exp(sigma * t)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    pairs = [
        (m1, r'$e^{\sigma t}$  — simple real pole', 'steelblue'),
        (m2, r'$t\,e^{\sigma t}$  — repeated real pole', 'tomato'),
        (m3, r'$e^{\sigma t}\cos(\omega t)$  — complex pair', 'seagreen'),
    ]
    ylim = 12
    for ax, (y, title, col) in zip(axes, pairs):
        ax.plot(t, np.clip(y, -ylim, ylim), color=col, lw=2)
        ax.axhline(0, color='k', lw=0.5, ls='--')
        ax.set_ylim(-ylim, ylim)
        ax.set_xlabel('t', fontsize=11)
        ax.set_ylabel('Amplitude', fontsize=11)
        ax.set_title(title, fontsize=10)
        ax.grid(True, alpha=0.3)
        if np.any(np.abs(y) > ylim):
            ax.text(0.5, 0.95, '(clipped)', transform=ax.transAxes,
                    ha='center', fontsize=8, color='gray')

    axes[2].plot(t, np.clip( env, -ylim, ylim), 'k--', lw=1, alpha=0.35, label='\u00b1envelope')
    axes[2].plot(t, np.clip(-env, -ylim, ylim), 'k--', lw=1, alpha=0.35)
    axes[2].legend(fontsize=8)

    stab = 'stable (\u03c3<0)' if sigma < 0 else ('unstable (\u03c3>0)' if sigma > 0 else 'marginal (\u03c3=0)')
    fig.suptitle(f'CT Natural Modes  \u00b7  \u03c3 = {sigma:.1f},  \u03c9 = {omega:.1f}  \u00b7  {stab}',
                 fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.show()

interact(update_ct_modes,
    sigma=widgets.FloatSlider(min=-2, max=2, step=0.1, value=-0.5, description='\u03c3',
        continuous_update=False, style={'description_width': '20px'},
        layout=widgets.Layout(width='360px')),
    omega=widgets.FloatSlider(min=0.1, max=10, step=0.5, value=3.0, description='\u03c9',
        continuous_update=False, style={'description_width': '20px'},
        layout=widgets.Layout(width='360px')))

interactive(children=(FloatSlider(value=-0.5, continuous_update=False, description='σ', layout=Layout(width='3…

<function __main__.update_ct_modes(sigma, omega)>

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

def update_dt_modes(lam, theta):
    k = np.arange(0, 16)
    k_safe = np.maximum(k - 1, 0)
    m1 = lam**k
    with np.errstate(divide='ignore', invalid='ignore'):
        m2 = np.where(k == 0, 0.0, k * lam**k_safe)
    m2 = np.where(np.isfinite(m2), m2, 0.0)
    r = np.abs(lam)
    m3 = r**k * np.cos(theta * k)
    env = r**k

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    pairs = [
        (m1, r'$\lambda^k$  — simple real pole $z=\lambda$', 'steelblue'),
        (m2, r'$k\,\lambda^{k-1}$  — repeated real pole', 'tomato'),
        (m3, r'$r^k\cos(\theta k)$  — complex pair $|z|=r$', 'seagreen'),
    ]
    ylim = 12
    for ax, (y, title, col) in zip(axes, pairs):
        yc = np.clip(y, -ylim, ylim)
        ax.vlines(k, 0, yc, color=col, alpha=0.6, lw=1.5)
        ax.scatter(k, yc, color=col, zorder=3, s=30)
        ax.axhline(0, color='k', lw=0.5, ls='--')
        ax.set_ylim(-ylim, ylim)
        ax.set_xlabel('k', fontsize=11)
        ax.set_ylabel('Amplitude', fontsize=11)
        ax.set_title(title, fontsize=10)
        ax.grid(True, alpha=0.3)
        if np.any(np.abs(y) > ylim):
            ax.text(0.5, 0.95, '(clipped)', transform=ax.transAxes,
                    ha='center', fontsize=8, color='gray')

    axes[2].plot(k, np.clip( env, -ylim, ylim), 'k--', lw=1, alpha=0.35, label='\u00b1envelope')
    axes[2].plot(k, np.clip(-env, -ylim, ylim), 'k--', lw=1, alpha=0.35)
    axes[2].legend(fontsize=8)

    stab = f'stable (|\u03bb|={r:.2f}<1)' if r < 1 else (f'unstable (|\u03bb|={r:.2f}>1)' if r > 1 else 'marginal (|\u03bb|=1)')
    fig.suptitle(f'DT Natural Modes  \u00b7  \u03bb = {lam:.2f},  \u03b8 = {theta:.2f} rad/sample  \u00b7  {stab}',
                 fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.show()

interact(update_dt_modes,
    lam=widgets.FloatSlider(min=-1.5, max=1.5, step=0.05, value=0.8, description='\u03bb',
        continuous_update=False, style={'description_width': '20px'},
        layout=widgets.Layout(width='360px')),
    theta=widgets.FloatSlider(min=0, max=6.28, step=0.1, value=1.0, description='\u03b8 (rad)',
        continuous_update=False, style={'description_width': '55px'},
        layout=widgets.Layout(width='360px')))

interactive(children=(FloatSlider(value=0.8, continuous_update=False, description='λ', layout=Layout(width='36…

<function __main__.update_dt_modes(lam, theta)>